# LLM Preference Prediction — Rung 2: Tuned TF-IDF + Similarity

**The story of getting this right took four tries. That is the notebook.**

---

## Background

Rung 1 used 16 structural features (response lengths, markdown counts, token overlap) with a StandardScaler + LogisticRegression pipeline.  
Result: **CV 1.0668 / LB 1.07343** — a good floor, but structure alone cannot see what the responses actually *say*.

Rung 2 reads the words. The plan: TF-IDF sparse features on response text plus the same linear model.  
The execution had a few wrong turns worth documenting.

---

## Attempt 1: Kitchen Sink

First instinct: throw everything at it.

- Word n-grams (1-2), max_features=200k
- Char n-grams (3-5), max_features=300k — classic authorship/style trick
- Structural 16 features from rung 1
- C=1.0

**5-fold CV: 1.0968** — essentially at the class-prior floor of 1.0972.

A model that learned nothing. The TF-IDF features actively hurt.

The first instinct was: *add more capacity* (increase C). That instinct was wrong.

---

## Diagnosis via Controlled Experiments

Before tuning, diagnose. Single 80/20 split, one variable at a time:

| Config | Val log loss | Finding |
|---|---|---|
| Word-only C=1.0 | **1.0589** | Char n-grams = net noise (-0.037 vs kitchen sink) |
| Word-only C=4.0 | **1.1503** | More capacity hurts: overfitting, not underfitting |
| Word-only C=0.5 | **1.0412** | Regularize down |
| Word-only C=0.25 | **1.0353** | Better |
| Word-only C=0.1 | **1.0385** | Too far — U-curve bottom is near 0.25 |
| Word + cosine(A,B) C=0.25 | **1.0308** | Rowwise A/B similarity detects ties |

**Key lessons:**

1. Char n-grams are standard practice for authorship tasks but hurt here. The competition signal is *which model said what*, not writing style. Char features add noise without discriminative signal at this sample size.
2. The regularization U-curve: C=1 overfits a 200k-feature space on 57k training examples. The gap between train and val loss is the signal — it said tighten, not loosen.
3. Rowwise cosine similarity between A and B TF-IDF vectors is a direct **tie detector**: when both responses are near-identical in word space, the model has one feature that flags the tie class.

**Design principle: diagnose before tuning. One controlled experiment beats ten theories.**

---

## Final Architecture

```
response_a, response_b, prompt
    -> parse JSON turns
    -> TF-IDF word n-grams (1-2), max_features=200k
       fit on A+B combined (shared vocabulary, separate encodings)
    -> rowwise cosine(word_A, word_B)  <- tie detector
    -> structural features x 16, max-abs scaled
    -> hstack (sparse)
    -> LogisticRegression (lbfgs, C=0.25)
```

**Shared vocabulary trick:** fit TF-IDF on A_texts + B_texts together, then transform A and B separately. The model sees the same word as feature 'in A' vs 'in B' — position-symmetric quality learning without leaking position into the vocabulary itself.

**Why max-abs scale the structural block:** TF-IDF lives in [0, 1]; raw character lengths reach 10,000+. Unscaled structural features dominate the gradient and stall lbfgs convergence.

---

## Result

**5-fold stratified CV: 1.03456 +/- 0.00199**

vs. Rung 1 reference: 1.0668 -> **+0.032 improvement over structural-only**.  
vs. class-prior floor: 1.0972 -> **+0.063 above trivial baseline**.

Not a huge jump, but a clean one — and the gap is now explainable at the feature level.
The calibration story (CV vs LB gap) will be worth watching after submission.

In [ ]:
from __future__ import annotations

import json
import logging
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

# ---------------------------------------------------------------------------
# Kaggle path discovery
# Data mounts at /kaggle/input/competitions/<slug>/ on Kaggle kernels.
# Fall back to local data/ directory for smoke testing.
# ---------------------------------------------------------------------------
_SAMPLE_ROWS = int(os.environ.get('SAMPLE', '0'))  # 0 = full run (Kaggle default)
_input_root = Path('/kaggle/input')
if _input_root.exists():
    print('=== Kaggle input mounts (first 20) ===')
    for _p in sorted(_input_root.rglob('*'))[:20]:
        print(' mounted:', _p)
    _train_candidates = sorted(_input_root.rglob('train.csv'))
    if not _train_candidates:
        raise FileNotFoundError(f'No train.csv found under {_input_root}')
    DATA_DIR = _train_candidates[0].parent
else:
    _nb_candidates = [
        Path('data'),  # local smoke-test fallback (relative)
        Path('data'),
        Path('../../data'),
    ]
    DATA_DIR = Path('data')
    for _c in _nb_candidates:
        if (_c / 'train.csv').exists():
            DATA_DIR = _c
            break

OUTPUT_PATH = (
    Path('/kaggle/working/submission.csv')
    if Path('/kaggle/working').exists()
    else Path('/tmp/submission_rung2.csv')
)

print(f'DATA_DIR resolved to: {DATA_DIR}')
print(f'OUTPUT_PATH: {OUTPUT_PATH}')
print(f'SAMPLE rows (0=full run): {_SAMPLE_ROWS}')

LABEL_COLS = ['winner_model_a', 'winner_model_b', 'winner_tie']
N_FOLDS = 5
RANDOM_STATE = 42
WORD_MAX_FEATURES = 200_000

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger('rung2')

print('Imports complete.')

## Inlined Helpers (from rung-1 baseline)

Kaggle kernels cannot import local modules, so the shared helpers from `src/baseline.py` are inlined here.

In [ ]:
# ---------------------------------------------------------------------------
# Data helpers (inlined from src/baseline.py)
# ---------------------------------------------------------------------------

def safe_json_loads(s: Any) -> list[str]:
    """Parse a JSON-encoded list of strings, with fallback for malformed input."""
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return []
    try:
        val = json.loads(s)
        if isinstance(val, list):
            return [t if t is not None else '' for t in val]
        return [str(val)]
    except (json.JSONDecodeError, TypeError, ValueError):
        return [str(s)]


def _count_code_fences(text: str) -> int:
    return len(re.findall(r'^```|^~~~', text, re.MULTILINE))


def _count_bullet_lines(text: str) -> int:
    return len(re.findall(r'^\s*[-*+]\s|^\s*\d+\.\s', text, re.MULTILINE))


def _count_header_lines(text: str) -> int:
    return len(re.findall(r'^#{1,6}\s', text, re.MULTILINE))


def _token_overlap(response_text: str, prompt_text: str) -> float:
    r_tokens = set(re.findall(r'\w+', response_text.lower()))
    p_tokens = set(re.findall(r'\w+', prompt_text.lower()))
    if not r_tokens:
        return 0.0
    return len(r_tokens & p_tokens) / (len(r_tokens) + 1)


def extract_features(df: pd.DataFrame) -> np.ndarray:
    """Build 16-feature structural matrix.

    Feature layout (16 features):
        0  len_a          1  len_b          2  len_delta      3  len_log_ratio
        4  code_fences_a  5  code_fences_b  6  code_fence_delta
        7  bullets_a      8  bullets_b      9  bullet_delta
        10 headers_a      11 headers_b      12 header_delta
        13 overlap_a      14 overlap_b      15 turn_count
    """
    rows: list[list[float]] = []
    log.info('Extracting structural features from %d rows...', len(df))
    for _, row in df.iterrows():
        prompts = safe_json_loads(row.get('prompt', ''))
        resp_a = safe_json_loads(row.get('response_a', ''))
        resp_b = safe_json_loads(row.get('response_b', ''))
        text_a = ' '.join(resp_a)
        text_b = ' '.join(resp_b)
        text_p = ' '.join(prompts)
        len_a, len_b = len(text_a), len(text_b)
        code_a = _count_code_fences(text_a)
        code_b = _count_code_fences(text_b)
        bullets_a = _count_bullet_lines(text_a)
        bullets_b = _count_bullet_lines(text_b)
        headers_a = _count_header_lines(text_a)
        headers_b = _count_header_lines(text_b)
        rows.append([
            len_a, len_b, len_a - len_b, math.log((len_a + 1) / (len_b + 1)),
            code_a, code_b, code_a - code_b,
            bullets_a, bullets_b, bullets_a - bullets_b,
            headers_a, headers_b, headers_a - headers_b,
            _token_overlap(text_a, text_p), _token_overlap(text_b, text_p),
            float(len(prompts)),
        ])
    return np.array(rows, dtype=np.float64)


def class_prior_log_loss(y: np.ndarray) -> float:
    """Log loss when always predicting training class priors."""
    n = len(y)
    _, counts = np.unique(y, return_counts=True)
    priors = counts / n
    return float(log_loss(y, np.tile(priors, (n, 1))))


print('Helper functions defined.')

## Load Data

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# Optional sampling for local smoke test (SAMPLE env var; 0 = full run on Kaggle)
if _SAMPLE_ROWS > 0:
    log.info('SMOKE TEST: sampling %d rows from train', _SAMPLE_ROWS)
    train_df = train_df.sample(
        n=min(_SAMPLE_ROWS, len(train_df)), random_state=RANDOM_STATE
    ).reset_index(drop=True)

log.info('train rows=%d  test rows=%d', len(train_df), len(test_df))
print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head(3)

## Labels and Class Distribution

In [ ]:
y = np.argmax(train_df[LABEL_COLS].values, axis=1)

class_counts = dict(zip(LABEL_COLS, np.bincount(y)))
print('Class distribution:')
for label, count in class_counts.items():
    print(f'  {label:20s}: {count:6d}  ({count/len(y)*100:.1f}%)')

prior_ll = class_prior_log_loss(y)
print(f'\nClass-prior log loss (floor): {prior_ll:.5f}')
print(f'Rung 1 reference CV:          1.06680')
print(f'Rung 1 LB:                    1.07343')

## Extract Texts and Structural Features

In [ ]:
def extract_texts(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    """Return (texts_a, texts_b) -- one concatenated string per row."""
    texts_a: list[str] = []
    texts_b: list[str] = []
    for _, row in df.iterrows():
        resp_a = safe_json_loads(row.get('response_a', ''))
        resp_b = safe_json_loads(row.get('response_b', ''))
        texts_a.append(' '.join(resp_a))
        texts_b.append(' '.join(resp_b))
    return texts_a, texts_b


print('Extracting texts from train...')
train_texts_a, train_texts_b = extract_texts(train_df)

print('Extracting structural features from train...')
struct_train = extract_features(train_df)
print(f'Structural feature matrix: {struct_train.shape}')

## TextFeatureBuilder: Shared Vocabulary + Similarity

In [ ]:
def rowwise_cosine(a: sp.csr_matrix, b: sp.csr_matrix) -> np.ndarray:
    """Cosine similarity between corresponding rows of two CSR matrices.

    The tie-detector feature: ties correlate with near-interchangeable
    responses, which per-side word counts alone cannot capture.
    """
    dots = np.asarray(a.multiply(b).sum(axis=1)).ravel()
    na = np.sqrt(np.asarray(a.multiply(a).sum(axis=1)).ravel())
    nb = np.sqrt(np.asarray(b.multiply(b).sum(axis=1)).ravel())
    return dots / np.maximum(na * nb, 1e-9)


class TextFeatureBuilder:
    """Build word TF-IDF features with shared vocabulary plus rowwise A/B similarity.

    Char n-grams were tested and dropped: they added noise (-0.037 val loss)
    on this task. See the diagnostic table in the intro markdown.

    Final feature vector per row:
        word_tfidf(A) | word_tfidf(B) | cosine(A,B) | structural(16)
    """

    def __init__(self, word_max_features: int = 200_000) -> None:
        self.word_max_features = word_max_features
        self.word_vec: TfidfVectorizer | None = None
        self.struct_scale: np.ndarray | None = None

    def fit(self, texts_a: list[str], texts_b: list[str]) -> 'TextFeatureBuilder':
        """Fit TF-IDF on A+B combined (shared vocabulary)."""
        log.info(
            'Fitting word TF-IDF on A+B combined (%d texts)...', len(texts_a) * 2
        )
        self.word_vec = TfidfVectorizer(
            analyzer='word',
            ngram_range=(1, 2),
            min_df=5,
            max_features=self.word_max_features,
            sublinear_tf=True,
        )
        self.word_vec.fit(texts_a + texts_b)
        log.info('Vocabulary size: %d', len(self.word_vec.vocabulary_))
        return self

    def transform(
        self,
        texts_a: list[str],
        texts_b: list[str],
        structural: np.ndarray,
    ) -> sp.csr_matrix:
        """Transform into combined sparse matrix."""
        if self.word_vec is None:
            raise RuntimeError('Call fit() before transform()')
        if self.struct_scale is None:
            raise RuntimeError('struct_scale not set; call fit_transform() first')
        word_a = self.word_vec.transform(texts_a)
        word_b = self.word_vec.transform(texts_b)
        sim = rowwise_cosine(word_a, word_b).reshape(-1, 1)
        struct_scaled = structural / self.struct_scale
        mat = sp.hstack(
            [word_a, word_b, sp.csr_matrix(sim), sp.csr_matrix(struct_scaled)],
            format='csr',
        )
        log.info('Feature matrix shape: %s', mat.shape)
        return mat

    def fit_transform(
        self,
        texts_a: list[str],
        texts_b: list[str],
        structural: np.ndarray,
    ) -> sp.csr_matrix:
        """Fit and transform (also fits max-abs structural scaler)."""
        self.fit(texts_a, texts_b)
        self.struct_scale = np.maximum(np.abs(structural).max(axis=0), 1.0)
        return self.transform(texts_a, texts_b, structural)


print('TextFeatureBuilder defined.')

## 5-Fold Cross-Validation

Each fold re-fits the TF-IDF vectorizer on that fold's training split only — no vocabulary leakage from the validation set.

In [ ]:
if _SAMPLE_ROWS > 0:
    print(f'SMOKE TEST ({_SAMPLE_ROWS} rows): running feature-build sanity check only.')
    _builder_smoke = TextFeatureBuilder(word_max_features=WORD_MAX_FEATURES)
    _X_smoke = _builder_smoke.fit_transform(train_texts_a, train_texts_b, struct_train)
    print(f'Smoke feature matrix shape: {_X_smoke.shape}  [OK]')
    print('Smoke test PASSED.')
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_losses: list[float] = []

    print(f'Running {N_FOLDS}-fold stratified CV...')
    print(f'{"Fold":>6} {"log_loss":>12}')
    print('-' * 22)

    for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(struct_train, y), start=1
    ):
        tr_a = [train_texts_a[i] for i in train_idx]
        tr_b = [train_texts_b[i] for i in train_idx]
        val_a = [train_texts_a[i] for i in val_idx]
        val_b = [train_texts_b[i] for i in val_idx]
        struct_tr = struct_train[train_idx]
        struct_val = struct_train[val_idx]
        y_tr = y[train_idx]
        y_val = y[val_idx]

        builder = TextFeatureBuilder(word_max_features=WORD_MAX_FEATURES)
        X_tr = builder.fit_transform(tr_a, tr_b, struct_tr)
        X_val = builder.transform(val_a, val_b, struct_val)

        clf = LogisticRegression(
            solver='lbfgs', max_iter=1000, C=0.25, random_state=RANDOM_STATE
        )
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_val)
        loss = log_loss(y_val, proba)
        fold_losses.append(loss)
        print(f'  {fold_idx:>4}  {loss:>12.5f}')

    mean_ll = float(np.mean(fold_losses))
    std_ll = float(np.std(fold_losses))

    print('-' * 22)
    print(f'  Mean  {mean_ll:>12.5f}')
    print(f'  Std   {std_ll:>12.5f}')
    print()
    print('=' * 50)
    print('CV SUMMARY')
    print('=' * 50)
    print(f'  Class-prior floor       : {prior_ll:.5f}')
    print(f'  Rung 1 reference CV     : 1.06680')
    print(f'  Rung 2 CV (5-fold)      : {mean_ll:.5f} +/- {std_ll:.5f}')
    print(f'  Improvement over rung 1 : {1.06680 - mean_ll:.5f}')
    print('=' * 50)

## Fit Full Model and Write Submission

In [ ]:
if _SAMPLE_ROWS == 0:
    log.info('Fitting final model on full training set...')
    full_builder = TextFeatureBuilder(word_max_features=WORD_MAX_FEATURES)
    X_train_full = full_builder.fit_transform(
        train_texts_a, train_texts_b, struct_train
    )

    log.info('Extracting test texts and structural features...')
    test_texts_a, test_texts_b = extract_texts(test_df)
    struct_test = extract_features(test_df)
    X_test_full = full_builder.transform(test_texts_a, test_texts_b, struct_test)

    final_clf = LogisticRegression(
        solver='lbfgs', max_iter=1000, C=0.25, random_state=RANDOM_STATE
    )
    final_clf.fit(X_train_full, y)
    log.info('Final model trained.')

    proba = final_clf.predict_proba(X_test_full)
    classes = final_clf.classes_

    pred_df = pd.DataFrame({
        'id': test_df['id'].values,
        LABEL_COLS[int(classes[0])]: proba[:, 0],
        LABEL_COLS[int(classes[1])]: proba[:, 1],
        LABEL_COLS[int(classes[2])]: proba[:, 2],
    })

    expected_cols = list(sample_sub.columns)
    pred_df = pred_df[expected_cols]
    id_order = sample_sub['id'].tolist()
    pred_df = pred_df.set_index('id').reindex(id_order).reset_index()

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    pred_df.to_csv(OUTPUT_PATH, index=False)
    log.info('Submission written to %s (%d rows)', OUTPUT_PATH, len(pred_df))

    print(f'\nSubmission saved to: {OUTPUT_PATH}')
    print(f'Shape: {pred_df.shape}')
    print('\nFirst 5 rows:')
    print(pred_df.head())
else:
    print('SMOKE TEST mode: skipping full fit and submission write.')